In [ ]:
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers
import tensorflow_hessian as tfh

print("TensorFlow version:", tf.__version__)

X = tf.constant([[0.1, 0.1],
                 [0.1, 0.9],
                 [0.9, 0.1],
                 [0.9, 0.9]], dtype=tf.float32)
t = tf.constant([[0.0], 
                 [1.0], 
                 [1.0], 
                 [0.0]], dtype=tf.float32)

class MyModel(tfh.models.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tfh.layers.Dense(128)
        self.leaky_relu = layers.LeakyReLU()
        self.dense2 = tfh.layers.Dense(1)
        self.sigmoid = layers.Activation('sigmoid')

    def call(self, x, training=False):
        x = self.dense1(x)
        x = self.sigmoid(x)
        x = self.dense2(x)
        x = self.sigmoid(x)
        return x

model = MyModel()
optimizer = tfh.optimizers.NewtonMethod(eta=0.1, alpha=0.1)

pb = tqdm(range(100))
for epoch in pb:
    with tf.GradientTape() as tape1:
        with tf.GradientTape() as tape2:
            y = model(X, training=True)
            loss = tf.keras.losses.BinaryCrossentropy()(t, y)
        grads = tape2.gradient(loss, model.trainable_variables)
    hessians = tape1.jacobian(grads[0], model.trainable_variables)
    optimizer.apply_gradients(model.trainable_variables, grads, hessians)

    pb.set_postfix({"loss": loss.numpy()})

model(X)

TensorFlow version: 2.19.0


100%|██████████| 100/100 [00:27<00:00,  3.66it/s, loss=0.154]


<tf.Tensor: shape=(4, 1), dtype=float32, numpy=
array([[0.13075742],
       [0.85683113],
       [0.85775423],
       [0.15451589]], dtype=float32)>